# Krea2 on Kaggle (ComfyUI, safetensors)

Deploy ComfyUI (latest master) + Ray Artshoot Krea2 v5 nvfp4 (safetensors, UNETLoader) + Qwen3-VL 4B fp8 encoder + Qwen Image VAE.

Ray Artshoot v5 nvfp4 is 7.2 GB, so the standard UNETLoader route fits: Ray 7.2 GB + encoder 5.24 GB + VAE 0.25 GB = ~12.7 GB, inside both /kaggle/working (20 GB) and T4 VRAM (16 GB). The Ray model reaches Kaggle as the private dataset kitzhang2019/ray-artshoot-krea2-v5-nvfp4 attached to this kernel (the App mounts it in runJob), located by a recursive search under /kaggle/input (the real mount path varies, never hard-code it), then copied into ComfyUI's diffusion_models. This mirrors the Moody Moss Mix V3.3 dataset pattern used by the MOSS notebook.

Sampling: 8 steps, cfg 1, ddim, beta via KSamplerAdvanced (add_noise enable, start 0, end 8, return_with_leftover_noise enable) -- the local Herraw workflow sampler. HerRAW LoRA 0.3 by default. Negative is ConditioningZeroOut (the positive conditioning zeroed out), as in the template.

Note: two-stage generation (2026-11-15, synced to the station/disk template) -- base image at half the final size, then SeedVR2 7B upscale to the final size, plus a WebP fine export (q95/method 6, 20MB cap).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!cd /kaggle/working/ComfyUI && grep -v -E "^(torch|torchvision|torchaudio)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/comfy_reqs.txt && pip install -q -r /tmp/comfy_reqs.txt
!pip install -q torchsde
!python3 -c "import torchsde; print('torchsde ok')"
!ls /kaggle/working/ComfyUI | head -20

# SeedVR2 plugin (numz, 2026-11-15): the SeedVR2VideoUpscaler node upscales the
# low-res generation to 2048x3072 in phase 2. Same install as the verified 7B test.
!if [ -d /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler ]; then echo seedvr-plugin-exists; else git clone --depth 1 https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler; fi
!cd /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler && grep -v -E "^(torch|torchvision)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/seedvr_reqs.txt && pip install -q -r /tmp/seedvr_reqs.txt
!ls /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler | head -20


In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
#
# LoRA (2026-11-15, user decision: HerRAW 0.3 on by default): HerRAW Sensual v0.2
# (Civitai 2863032, version 3338894), strength 0.3 (local workflow
# Herraw-V01-Krea2-v10.json: LoraLoaderModelOnly, 0.3). HF mirror
# RunningHubAI/rh-herraw-sensual-v0.2-lora hosts the same file.
# patchNotebook clears LORA_NAME when the form toggle is off.
LORA_NAME = "HerRAW_Sensual_v0.2.safetensors"
LORA_REPO = "RunningHubAI/rh-herraw-sensual-v0.2-lora"
LORA_STRENGTH = 0.3
# Krea2 files (2026-11-15: main model synced to Ray, same as station/disk template):
# - Ray Artshoot v5 nvfp4 7.2 GB (Civitai 2574319, version 3378023, file 3318813;
#   reaches Kaggle as the private dataset kitzhang2019/ray-artshoot-krea2-v5-nvfp4,
#   located by recursive search under /kaggle/input, then copied into diffusion_models)
# - encoder qwen3vl_4b_fp8_scaled 5.24 GB (Comfy-Org/Krea-2, text_encoders/)
# - VAE qwen_image_vae 0.25 GB (Comfy-Org/Krea-2, vae/)
# Total ~12.7 GB, fits /kaggle/working 20 GB and T4 16 GB VRAM.
import glob, shutil
RAY_NAME = "Ray-Krea2-Artshoot-v5-turbo-nvfp4.safetensors"
hits = sorted(glob.glob("/kaggle/input/**/" + RAY_NAME, recursive=True))
if not hits:
    raise SystemExit("ray model not found under /kaggle/input: dataset kitzhang2019/ray-artshoot-krea2-v5-nvfp4 is not attached to this kernel")
ray_dst = os.path.join(base, "diffusion_models", RAY_NAME)
os.makedirs(os.path.dirname(ray_dst), exist_ok=True)
shutil.copyfile(hits[0], ray_dst)
print("ray model:", hits[0], "->", ray_dst, round(os.path.getsize(ray_dst) / 1e9, 2), "GB")
for fname in ["text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "vae/qwen_image_vae.safetensors"]:
    p = hf_hub_download(repo_id="Comfy-Org/Krea-2", filename=fname, local_dir=base)
    print(fname, round(os.path.getsize(p) / 1e9, 2), "GB")
if LORA_NAME:
    p = hf_hub_download(repo_id=LORA_REPO, filename=LORA_NAME, local_dir=base + "/loras")
    print("lora", LORA_NAME, round(os.path.getsize(p) / 1e9, 2), "GB")
# Guard: the three files must sit exactly where the loaders look, or fail now.
expected = {
    "diffusion_models": RAY_NAME,
    "text_encoders": "qwen3vl_4b_fp8_scaled.safetensors",
    "vae": "qwen_image_vae.safetensors",
}
for folder, fname in expected.items():
    fp = os.path.join(base, folder, fname)
    print(folder + "/", fname, "exists:", os.path.exists(fp))
    assert os.path.exists(fp), "missing model file: " + fp

# SeedVR2 7B (2026-11-15): DiT seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors
# (7.88 GB, AInVFX/SeedVR2_comfyUI) + VAE ema_vae_fp16.safetensors (0.47 GB,
# numz/SeedVR2_comfyUI) come from the private dataset kitzhang2019/seedvr2-7b
# attached to this kernel, located by recursive search under /kaggle/input.
# They are SYMLINKED (not copied) into ComfyUI/models/SEEDVR2/: copying all
# 8.35 GB would overflow /kaggle/working (Qwen already uses ~17 GB of 20 GB).
# Symlinks read through to /kaggle/input, which stays mounted for the run.
import glob as _seedvr_glob
SEEDVR_DIT = "seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors"
SEEDVR_VAE = "ema_vae_fp16.safetensors"
seedvr_dir = os.path.join(base, "SEEDVR2")
os.makedirs(seedvr_dir, exist_ok=True)
for _name in (SEEDVR_DIT, SEEDVR_VAE):
    _hits = sorted(_seedvr_glob.glob("/kaggle/input/**/" + _name, recursive=True))
    if not _hits:
        raise SystemExit("seedvr file not found under /kaggle/input: " + _name + " (dataset kitzhang2019/seedvr2-7b is not attached to this kernel)")
    _link = os.path.join(seedvr_dir, _name)
    if os.path.islink(_link) or os.path.exists(_link):
        os.remove(_link)
    os.symlink(_hits[0], _link)
    print("seedvr:", _hits[0], "->", _link, round(os.path.getsize(_link) / 1e9, 2), "GB")
# Guard: the two files must be visible exactly where the loaders look.
for _name in (SEEDVR_DIT, SEEDVR_VAE):
    _fp = os.path.join(seedvr_dir, _name)
    print("SEEDVR2/", _name, "exists:", os.path.exists(_fp))
    assert os.path.exists(_fp), "missing seedvr file: " + _fp
print("seedvr models ready")
print("all models downloaded")


In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = "A photorealistic portrait of a young woman in a sunlit garden, soft morning light, shallow depth of field, 50mm lens"
# Loader: standard UNETLoader (safetensors). Ray Artshoot v5 nvfp4 is a
# safetensors checkpoint (2026-11-15: App synced to the station/disk Ray config).
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
need = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "ConditioningZeroOut",
        "EmptyLatentImage", "KSamplerAdvanced", "VAEDecode", "SaveImage", "LoadImage", "SeedVR2LoadDiTModel", "SeedVR2LoadVAEModel", "SeedVR2VideoUpscaler"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", { }))
    return {k: v for k, v in preferred.items() if k in allowed}

# Print the loader + sampler specs to the log so a wrong input name/value is caught here, not after a long run.
for ct in ["UNETLoader", "CLIPLoader", "KSamplerAdvanced"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:800])

# Raw-node version of the official Krea2 Turbo T2I template (subgraph expanded,
# prompt-enhancement branch dropped, prompt sent as-is). Core pipeline:
# UNETLoader (Ray nvfp4 safetensors) -> CLIPLoader (type krea2) -> VAELoader; CLIPTextEncode ->
# ConditioningZeroOut (negative); EmptyLatentImage 1024x1024;
# KSamplerAdvanced 8 steps, cfg 1, ddim, beta, add_noise enable, start 0, end 8,
# return_with_leftover_noise enable (weave fix 2026-11-15, disk template); VAEDecode; SaveImage.
nodes = {}
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "Ray-Krea2-Artshoot-v5-turbo-nvfp4.safetensors", "weight_dtype": "default"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "qwen_image_vae.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}}
nodes["6"] = {"class_type": "EmptyLatentImage", "inputs": node_inputs("EmptyLatentImage", {"width": 1024, "height": 1024, "batch_size": 1})}
# LoRA (optional): when LORA_NAME is set, patch the model with LoraLoaderModelOnly
# (model-only, no CLIP) between the UNETLoader and KSampler. When empty, the
# model goes straight from the UNETLoader to KSampler, unchanged.
MODEL_OUT = ["1", 0]
if LORA_NAME:
    nodes["20"] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": ["1", 0], "lora_name": LORA_NAME, "strength_model": LORA_STRENGTH})}
    MODEL_OUT = ["20", 0]
nodes["7"] = {"class_type": "KSamplerAdvanced", "inputs": node_inputs("KSamplerAdvanced", {"model": MODEL_OUT, "add_noise": "enable", "noise_seed": 42, "steps": 8, "cfg": 1, "sampler_name": "ddim", "scheduler": "beta", "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "start_at_step": 0, "end_at_step": 8, "return_with_leftover_noise": "enable", "denoise": 1.0})}
nodes["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
nodes["9"] = {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "krea2_lowres"}}

# Pre-check: every required input of every node must be present, or fail here with the
# full list (do not let the server reject it one input at a time). Plus a combo-value
# check: where the node definition lists the allowed values, the value we send must be
# one of them (a bad value can be silently dropped and only fail at execution -- the Wan
# V5 lesson). NOTE: a combo's allowed values live in cfg[0] as a LIST; a scalar input
# (STRING/INT/FLOAT) has cfg[0] as a type-name string, so it is skipped -- this is the
# exact check the Qwen notebooks use, and the Krea2 V1 failure was a wrong simplified
# version of it that treated the free-text prompt as a combo value.
problems = []
for nid, node in nodes.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        if isinstance(cfg, list) and cfg and isinstance(cfg[0], list) and isinstance(val, (str, int, float)):
            if val not in cfg[0]:
                problems.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(cfg[0]))
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

# ---------- SeedVR helpers (from the verified 7B test notebook) ----------
def combo_opts_for_cfg(cfg):
    """Options list for a combo input spec, or None if the input is not a combo.
    Old format: [["opt1", "opt2"], {...}]. New format (current ComfyUI master /
    SeedVR2 plugin): ["COMBO", {"options": [...], "default": ..., ...}]."""
    if not isinstance(cfg, list) or not cfg:
        return None
    if isinstance(cfg[0], list):
        return cfg[0]
    if cfg[0] == "COMBO" and len(cfg) > 1 and isinstance(cfg[1], dict) and "options" in cfg[1]:
        return cfg[1]["options"]
    return None

def combo_options(class_type, name):
    """Return the option list for a combo input (both spec formats)."""
    spec = info[class_type]["input"]
    cfg = spec.get("required", {}).get(name) or spec.get("optional", {}).get(name)
    opts = combo_opts_for_cfg(cfg)
    return opts if opts is not None else ([cfg[0]] if cfg else [])

# Print the SeedVR2 node specs so a wrong input name/value is caught here, not after a long run.
for ct in ["SeedVR2LoadDiTModel", "SeedVR2LoadVAEModel", "SeedVR2VideoUpscaler"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:1200])

# Resolve device + model values from the live specs (never hard-code a combo value).
dit_device = combo_options("SeedVR2LoadDiTModel", "device")[0]
vae_device = combo_options("SeedVR2LoadVAEModel", "device")[0]
dit_models = combo_options("SeedVR2LoadDiTModel", "model")
vae_models = combo_options("SeedVR2LoadVAEModel", "model")
print("dit device:", dit_device, "| vae device:", vae_device)
print("dit models available:", dit_models)
print("vae models available:", vae_models)
DIT_MODEL = "seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors"
VAE_MODEL = "ema_vae_fp16.safetensors"
assert DIT_MODEL in dit_models, "DiT model not in loader options: " + DIT_MODEL
assert VAE_MODEL in vae_models, "VAE model not in loader options: " + VAE_MODEL

# ---------- Phase 2 nodes: SeedVR2 7B upscale (1024x1536 -> 2048x3072) ----------
# Verified 2026-11-15 (kernel kitzhang2019/seedvr-7b-ddimbeta-test-20261114):
# resolution 2048, max 6000, batch 5, blocks_to_swap 36, VAE tiled 1024/128,
# sdpa attention, color lab. 390 s on T4; 4096 OOMs in DiT attention.
nodes_seedvr = {}
nodes_seedvr["31"] = {"class_type": "LoadImage", "inputs": node_inputs("LoadImage", {"image": "krea2_lowres.png"})}
nodes_seedvr["32"] = {"class_type": "SeedVR2LoadDiTModel", "inputs": node_inputs("SeedVR2LoadDiTModel", {
    "model": DIT_MODEL, "device": dit_device, "blocks_to_swap": 36,
    "swap_io_components": False, "offload_device": "cpu", "attention_mode": "sdpa"})}
nodes_seedvr["33"] = {"class_type": "SeedVR2LoadVAEModel", "inputs": node_inputs("SeedVR2LoadVAEModel", {
    "model": VAE_MODEL, "device": vae_device, "encode_tiled": True, "encode_tile_size": 1024,
    "encode_tile_overlap": 128, "decode_tiled": True, "decode_tile_size": 1024,
    "decode_tile_overlap": 128, "tile_debug": "false", "offload_device": "cpu"})}
nodes_seedvr["34"] = {"class_type": "SeedVR2VideoUpscaler", "inputs": node_inputs("SeedVR2VideoUpscaler", {
    "image": ["31", 0], "dit": ["32", 0], "vae": ["33", 0], "seed": 42,
    "resolution": 2048, "max_resolution": 6000, "batch_size": 5,
    "uniform_batch_size": False, "temporal_overlap": 0, "prepend_frames": 0,
    "color_correction": "lab", "input_noise_scale": 0.0})}
nodes_seedvr["35"] = {"class_type": "SaveImage", "inputs": {"images": ["34", 0], "filename_prefix": "krea2_test"}}


def submit_and_wait(nodes, save_node_id, out_path, timeout_s, label):
    """Submit one prompt, poll until success/error/timeout, download the SaveImage output. Returns (done, elapsed_s)."""
    r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
    print(label, "submit:", r.status_code, r.text[:400])
    r.raise_for_status()
    pid = r.json()["prompt_id"]
    print(label, "prompt_id:", pid)
    t0 = time.time()
    done = False
    while time.time() - t0 < timeout_s:
        h = requests.get(BASE + "/history/" + pid, timeout=60).json()
        if pid in h:
            st = h[pid].get("status", {})
            sstr = st.get("status_str")
            if sstr == "success":
                outs = h[pid]["outputs"]
                imgs = outs.get(save_node_id, {}).get("images", [])
                print(label, "images:", imgs)
                if imgs:
                    im = imgs[0]
                    q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                    data = requests.get(BASE + "/view", params=q, timeout=300).content
                    open(out_path, "wb").write(data)
                    print(label, "saved", out_path, len(data), "bytes")
                    done = True
                break
            if sstr == "error":
                print(label, "RUN ERROR:", json.dumps(h[pid], indent=2)[:3000])
                break
        time.sleep(15)
    elapsed = round(time.time() - t0)
    print(label, "done:", done, "elapsed:", elapsed, "s")
    return done, elapsed

# Phase 1: low-res generation (1024x1536). Krea2 8 steps, expect a few min; allow 60 min.
gen_ok, gen_s = submit_and_wait(nodes, "9", "/kaggle/working/krea2_lowres.png", 3600, "GEN")
if not gen_ok:
    proc.terminate()
    raise SystemExit("phase 1 (generation) failed")

# LoadImage reads from ComfyUI/input/; copy the low-res output there.
import shutil
os.makedirs("/kaggle/working/ComfyUI/input", exist_ok=True)
shutil.copyfile("/kaggle/working/krea2_lowres.png", "/kaggle/working/ComfyUI/input/krea2_lowres.png")
print("copied low-res to ComfyUI/input/krea2_lowres.png")

# Re-fetch object_info now that the low-res file is in ComfyUI/input/:
# the LoadImage combo options are snapshotted from the input dir at fetch
# time, so the pre-check must see the file Phase 1 just produced.
info = requests.get(BASE + "/object_info", timeout=120).json()

# Pre-check for the SeedVR nodes: every required input present, and every
# combo value one of the live options (same check as the generation phase).
problems_sv = []
for nid, node in nodes_seedvr.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems_sv.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        opts = combo_opts_for_cfg(cfg)
        if opts is not None and isinstance(val, (str, int, float)):
            if val not in opts:
                problems_sv.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(opts))
print("seedvr missing required inputs:", problems_sv if problems_sv else "none")
if problems_sv:
    proc.terminate()
    raise SystemExit("seedvr missing required inputs: " + ", ".join(problems_sv))

# Phase 2: SeedVR2 7B upscale (1024x1536 -> 2048x3072). Verified ~390 s; allow 30 min.
sv_ok, sv_s = submit_and_wait(nodes_seedvr, "35", "/kaggle/working/krea2_test.png", 1800, "SEEDVR")
if not sv_ok:
    proc.terminate()
    raise SystemExit("phase 2 (SeedVR upscale) failed")

# Phase 3: WebP fine export (quality 95 / method 6; step down to 90, then 85 only
# if still over 20MB). The 2048x3072 PNG is ~8.8MB, over the station's 8MB
# return cap, so the WebP is the file the station fetches back.
from PIL import Image as _WImage
def webp_fine(png_path, webp_path):
    sz = None
    q = None
    for _q in (95, 90, 85):
        _WImage.open(png_path).convert("RGB").save(webp_path, "WEBP", quality=_q, method=6)
        _sz = os.path.getsize(webp_path)
        print("webp q%d" % _q, webp_path, _sz, "bytes")
        sz, q = _sz, _q
        if _sz <= 20 * 1024 * 1024 or _q == 85:
            break
    return sz, q

webp_s, webp_q = webp_fine("/kaggle/working/krea2_test.png", "/kaggle/working/krea2_test.webp")

proc.terminate()
print("server stopped")
print("RESULT gen_s=%d seedvr_s=%d webp_bytes=%d webp_q=%d" % (gen_s, sv_s, webp_s, webp_q))
print("exists:", os.path.exists("/kaggle/working/krea2_test.webp"), "bytes:", os.path.getsize("/kaggle/working/krea2_test.webp") if os.path.exists("/kaggle/working/krea2_test.webp") else 0)
assert os.path.exists("/kaggle/working/krea2_test.webp"), "no output webp: /kaggle/working/krea2_test.webp"


In [ ]:
from IPython.display import Image, display
import os
# NOTE: do NOT display the .webp here -- IPython cannot embed webp
# ("Cannot embed the 'webp' image format") and papermill marks the whole
# kernel ERROR even though generation succeeded. Only the PNG is displayed.
for p in ["/kaggle/working/krea2_test.png", "/kaggle/working/krea2_test.webp"]:
    print(p, "exists:", os.path.exists(p), "bytes:", os.path.getsize(p) if os.path.exists(p) else 0)
if os.path.exists("/kaggle/working/krea2_test.png"):
    display(Image(filename="/kaggle/working/krea2_test.png"))
